In [1]:
!pip install streamlit -q
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 95.1 MB/s eta 0:00:00
/kaggle/input/notebooks/tanishqraina/plantvillage/__results__.html
/kaggle/input/notebooks/tanishqraina/plantvillage/__notebook__.ipynb
/kaggle/input/notebooks/tanishqraina/plantvillage/__output__.json
/kaggle/input/notebooks/tanishqraina/plantvillage/custom.css
/kaggle/input/notebooks/tanishqraina/plantvillage/__results___files/__results___12_36.png
/kaggle/input/notebooks/tanishqraina/plantvillage/__results___files/__results___10_35.png
/kaggle/input/notebooks/tanishqraina/plantvillage/__results___files/__results___12_35.png
/kaggle/input/datasets/farukalam/tomato-leaf-diseases-detection-computer-vision/README.dataset.txt
/kaggle/input/datasets/farukalam/tomato-leaf-diseases-detection-computer-vision/README.roboflow.txt
/kaggle/input/datasets/farukalam/tomato-leaf-diseases-detection-computer-vision/data.yaml
/kaggle/input/datase

Build a Convolutional Neural Network from scratch to classify images of plant leaves as either "Healthy" or "Diseased".

In [2]:
import torch
import torch.nn as nn
import torchvision.datasets
import torch.nn.functional as F

In [3]:

class LeafNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(3,16,5)
        self.pool1=nn.MaxPool2d(2,2)
        self.conv2=nn.Conv2d(16,32,5)
        self.pool2=nn.MaxPool2d(2,2)
        self.fc1=nn.Linear(32*29*29,128)
        self.fc2=nn.Linear(128,2)
    def forward(self,x):
        x=self.pool1(F.relu(self.conv1(x)))
        x=self.pool2(F.relu(self.conv2(x)))
        x=x.view(-1,32*29*29)
        x=F.relu(self.fc1(x))
        return self.fc2(x)
model=LeafNet()
print(model)

LeafNet(
  (conv1): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=26912, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)


In [4]:
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
# 1. Define Transforms (Resize to 128x128 for our LeafNet)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Load Data (This downloads automatically)
train_data = datasets.ImageFolder(
    root='/kaggle/input/datasets/ajayrana/hymenoptera-data/hymenoptera_data/train/', 
    transform=transform
)

# 3. Create DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

# 4. Check Classes
print(f"Classes: {train_data.classes}")
print(f"Total Images: {len(train_data)}")

Classes: ['ants', 'bees']
Total Images: 244


In [5]:
model=LeafNet()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)
epochs=5
for epoch in range(5):
    running_loss=0.0
    for images,labels in train_loader:
        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs,labels)
        running_loss+=loss.item()
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/5], Loss: 1.3944
Epoch [2/5], Loss: 0.6884
Epoch [3/5], Loss: 0.6640
Epoch [4/5], Loss: 0.6184
Epoch [5/5], Loss: 0.5388


In [6]:
test_data=datasets.ImageFolder(
    root='/kaggle/input/datasets/ajayrana/hymenoptera-data/hymenoptera_data/val/',
    transform=transform
)
test_loader=DataLoader(test_data,batch_size=32,shuffle=True)
model.eval()
correct=0
total=0
with torch.no_grad():
    for images,labels in test_loader :
        outputs=model(images)
        _, predicted = torch.max(outputs.data, 1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
    print(100*correct/total)
torch.save(model.state_dict(), 'leaf_net.pth')

53.59477124183007


In [7]:
!pip install streamlit -q
!streamlit run app.py --server.port=8501 --server.headless=true

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [8]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as transforms

# 1. Define the Model Class (Must match your training code exactly)
class LeafNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 5)
        # Note: 29 comes from our math: (128 -> 124 -> 62 -> 58 -> 29)
        self.fc1 = nn.Linear(32 * 29 * 29, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32 * 29 * 29)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# 2. Load the Model
@st.cache_resource # Caches the model so it doesn't reload every time
def load_model():
    model = LeafNet()
    model.load_state_dict(torch.load('leaf_net.pth', map_location=torch.device('cpu')))
    model.eval()
    return model

model = load_model()

# 3. Define Transforms (Must match training)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 4. The UI
st.title("🌿 Plant Doctor AI")
st.write("Upload a leaf image to check if it's healthy or diseased.")

uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "png"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption='Uploaded Leaf', use_column_width=True)
    
    # Preprocess
    img_t = transform(image)
    batch_t = torch.unsqueeze(img_t, 0) # Add batch dimension
    
    # Predict
    with torch.no_grad():
        out = model(batch_t)
        _, predicted = torch.max(out, 1)
        
    classes = ['Ants', 'Bees'] # Or Healthy/Diseased
    st.success(f"Prediction: {classes[predicted.item()]}")

Writing app.py
